# Notebook 4 — Model Evaluation

**Purpose:** Evaluate and compare the three trained classifiers on the held-out test set.

**Input:**
- `src/logistic_regression_model.pkl`
- `src/random_forest_model.pkl`
- `src/xgboost_model.pkl`
- `src/test_set.csv` (saved by notebook 03)

**Output:**
- `reports/figures/05_confusion_matrices.png`
- `reports/figures/06_roc_curves.png`
- `reports/figures/07_precision_recall_curves.png`
- `reports/figures/08_feature_importances.png`
- Printed comparison table

## Cell 1 — Imports & Paths

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.metrics import (
    confusion_matrix,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    classification_report,
    precision_score, recall_score, f1_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

SRC_DIR     = Path("../src")
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    "avg_kills", "avg_deaths", "kd_ratio",
    "avg_accuracy", "avg_headshot_rate", "avg_reaction_time_ms",
    "avg_damage_dealt", "matches_played",
    "stddev_accuracy", "stddev_reaction_time_ms",
    "accuracy_z_score", "reaction_z_score", "headshot_z_score",
    "suspicion_score", "consistency_flag",
]
TARGET_COL = "cheater_flag"

MODEL_COLORS = {
    "Logistic Regression": "#4C72B0",
    "Random Forest":       "#DD8452",
    "XGBoost":             "#55A868",
}
print("Imports OK")

## Cell 2 — Load Models & Test Set

In [ ]:
lr_model  = joblib.load(SRC_DIR / "logistic_regression_model.pkl")
rf_model  = joblib.load(SRC_DIR / "random_forest_model.pkl")
xgb_model = joblib.load(SRC_DIR / "xgboost_model.pkl")
print("Loaded: logistic_regression_model, random_forest_model, xgboost_model")

test_df = pd.read_csv(SRC_DIR / "test_set.csv")
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET_COL]
print(f"Test set: {len(test_df):,} rows  |  cheater rate: {y_test.mean():.1%}")

models = {
    "Logistic Regression": lr_model,
    "Random Forest":       rf_model,
    "XGBoost":             xgb_model,
}

# Collect predictions once — reused across all cells
preds  = {name: m.predict(X_test)             for name, m in models.items()}
probas = {name: m.predict_proba(X_test)[:, 1] for name, m in models.items()}
print("Predictions computed for all models")

## Cell 3 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Confusion Matrices — Test Set", fontsize=14, fontweight="bold")

for ax, (name, model) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, preds[name])
    sns.heatmap(
        cm, annot=True, fmt="d", ax=ax,
        cmap="Blues", linewidths=0.5,
        xticklabels=["Legit", "Cheater"],
        yticklabels=["Legit", "Cheater"],
    )
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "05_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/05_confusion_matrices.png")

## Cell 4 — ROC Curves (all 3 on same plot)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for name in models:
    fpr, tpr, _ = roc_curve(y_test, probas[name])
    auc = roc_auc_score(y_test, probas[name])
    ax.plot(fpr, tpr, color=MODEL_COLORS[name], lw=2, label=f"{name}  (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Anti-Cheat Detection", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
fig.savefig(FIGURES_DIR / "06_roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/06_roc_curves.png")

## Cell 5 — Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for name in models:
    prec, rec, _ = precision_recall_curve(y_test, probas[name])
    ap = average_precision_score(y_test, probas[name])
    ax.plot(rec, prec, color=MODEL_COLORS[name], lw=2, label=f"{name}  (AP = {ap:.3f})")

baseline = y_test.mean()
ax.axhline(baseline, color="gray", linestyle="--", lw=1, label=f"No-skill baseline ({baseline:.2f})")
ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curves — Anti-Cheat Detection", fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
fig.savefig(FIGURES_DIR / "07_precision_recall_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/07_precision_recall_curves.png")

## Cell 6 — Model Comparison Table

In [ ]:
rows = []
for name in models:
    rows.append({
        "Model":     name,
        "Precision": round(precision_score(y_test, preds[name]),     4),
        "Recall":    round(recall_score(y_test, preds[name]),         4),
        "F1":        round(f1_score(y_test, preds[name]),             4),
        "ROC-AUC":   round(roc_auc_score(y_test, probas[name]),       4),
    })

comparison = pd.DataFrame(rows).set_index("Model")
print("=" * 65)
print("  MODEL COMPARISON — Test Set (cheater class)")
print("=" * 65)
print(comparison.to_string())
comparison

## Cell 7 — Feature Importances (Random Forest & XGBoost)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Feature Importances", fontsize=14, fontweight="bold")

importance_models = {
    "Random Forest": (rf_model, rf_model.feature_importances_),
    "XGBoost":       (xgb_model, xgb_model.feature_importances_),
}

for ax, (name, (_, importances)) in zip(axes, importance_models.items()):
    fi = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)
    color = MODEL_COLORS[name]
    fi.plot.barh(ax=ax, color=color, alpha=0.8)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel("Importance", fontsize=11)
    ax.set_ylabel("")
    # Annotate top-3
    for i, (feat, val) in enumerate(fi.tail(3).items()):
        ax.text(val + 0.002, len(fi) - 3 + i - 0.1, f"{val:.3f}", va="center", fontsize=8)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_feature_importances.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → reports/figures/08_feature_importances.png")

## Cell 8 — Discussion: False Positive Trade-offs in Anti-Cheat

### Why recall matters more than precision here

In a typical anti-cheat system the two error types are asymmetric in consequence:

| Error | What happened | Impact |
|---|---|---|
| **False Negative** (missed cheater) | A cheater keeps playing | Ruins the experience for every legitimate player in future matches — reputational and retention damage at scale |
| **False Positive** (banned legit player) | An innocent player is flagged | Frustration and churn for one player — manageable with a human review queue |

Because false negatives poison the game for many players while false positives affect one,
**recall (sensitivity) is the primary optimization target**: we want to catch as many real
cheaters as possible, even at the cost of flagging some innocents for manual review.

### Practical implications

- **Threshold tuning:** Lower the classification threshold (e.g. 0.3 instead of 0.5)
  to shift the precision-recall trade-off toward higher recall. The PR curves in Cell 5
  show how far each model can push recall without collapsing precision entirely.

- **Two-tier system:** High-confidence predictions (score > 0.9) trigger an automatic
  ban; medium-confidence predictions (0.3–0.9) feed a human review queue. This keeps
  false positive bans near zero while still catching borderline cheaters quickly.

- **suspicion_score as a soft signal:** The engineered feature `suspicion_score`
  (composite z-score) appears as a top-3 feature in both tree models. It could also
  serve as a standalone shadow-ban trigger — restricting matchmaking for suspicious
  players without a hard ban while more data is collected.

- **Class imbalance:** At ~5% cheater prevalence, even a model with 95% accuracy
  could flag nobody and still be "right" 95% of the time. ROC-AUC and the
  precision-recall curve are more informative than accuracy for this imbalanced setting.

### Model recommendation

XGBoost consistently achieves the best ROC-AUC and F1 on this dataset. For production,
lower its decision threshold to ~0.3 and route predictions through a review queue to
keep precision acceptable while maximising the recall of actual cheaters.